# Residuals

In [ ]:
library(tidyverse)
options(repr.matrix.max.rows = 20)

## Review: Residuals

$$\text{residual} = \text{observed value} - \text{regression estimate}$$

In [ ]:
original <- read_csv("data/family_heights.csv", show_col_types = FALSE)

heights <- tibble(
  MidParent = original$midparentHeight,
  Child = original$childHeight
)

family_model <- lm(Child ~ MidParent, data = heights)
coef(family_model)

In [ ]:
heights <- heights |>
  mutate(
    `Fitted Value` = fitted(family_model),
    Residual = residuals(family_model)
  )

heights

The same functions as last class: `scatter_fit()` and `residual_plot()`.

In [ ]:
scatter_fit <- function(table, x, y) {
  model <- lm(table[[y]] ~ table[[x]])
  ggplot(table, aes(x = .data[[x]], y = .data[[y]])) +
    geom_point(alpha = 0.5) +
    geom_abline(intercept = coef(model)[1], slope = coef(model)[2],
                color = "gold", linewidth = 2) +
    labs(x = x, y = y) +
    theme_minimal(base_size = 16)
}

residual_plot <- function(table, x, y) {
  model <- lm(table[[y]] ~ table[[x]])
  tibble(x_values = table[[x]], residuals = residuals(model)) |>
    ggplot(aes(x = x_values, y = residuals)) +
    geom_point(color = "red", alpha = 0.5) +
    geom_hline(yintercept = 0, color = "darkblue", linewidth = 2) +
    labs(x = x, title = "Residual Plot") +
    theme_minimal(base_size = 16)
}

In [ ]:
scatter_fit(heights, "MidParent", "Child")

In [ ]:
residual_plot(heights, "MidParent", "Child")

### Dugongs

Predict `Age` (years) from `Length` (meters).

In [ ]:
dugong <- read_csv("data/dugongs.csv", show_col_types = FALSE) |>
  select(Length, Age)

cor(dugong$Length, dugong$Age)

In [ ]:
scatter_fit(dugong, "Length", "Age")

In [ ]:
residual_plot(dugong, "Length", "Age")

## Part 1: A Line That Looks Perfect

`women` comes with R: the average weight (pounds) of American women at each height (inches).

In [ ]:
us_women <- as_tibble(women)
us_women

In [ ]:
ggplot(us_women, aes(x = height, y = weight)) +
  geom_point(size = 3) +
  theme_minimal(base_size = 16)

Guess $r$.

In [ ]:
cor(us_women$height, us_women$weight)

In [ ]:
scatter_fit(us_women, "height", "weight")

Is a line the right choice?

In [ ]:
residual_plot(us_women, "height", "weight")

How far off are the estimates?

In [ ]:
women_model <- lm(weight ~ height, data = us_women)

us_women <- us_women |>
  mutate(
    `Fitted Value` = fitted(women_model),
    Residual = residuals(women_model)
  )

us_women

## Part 2: Properties of Residuals

### The average of the residuals

In [ ]:
mean(heights$Residual)

In [ ]:
mean(us_women$Residual)

Dugongs and districts, from last class:

In [ ]:
dugong_model <- lm(Age ~ Length, data = dugong)
mean(residuals(dugong_model))

In [ ]:
demographics <- read_csv("data/district_demographics2016.csv", show_col_types = FALSE) |>
  select(`Median Income`, `College%`)

demographics_model <- lm(`Median Income` ~ `College%`, data = demographics)
mean(residuals(demographics_model))

Rounded:

In [ ]:
round(mean(residuals(demographics_model)), 6)

### The correlation between the residuals and $x$

In [ ]:
round(cor(heights$MidParent, heights$Residual), 6)

In [ ]:
round(cor(demographics$`College%`, residuals(demographics_model)), 6)

### The correlation between the residuals and the fitted values

In [ ]:
round(cor(heights$`Fitted Value`, heights$Residual), 6)

### Correlation 0 does not mean no pattern

The `us_women` residuals make a U shape.

In [ ]:
round(cor(us_women$height, us_women$Residual), 6)

## Part 3: Residuals From Other Lines

`other_line()` draws the residual plot for any line through the heights. It prints the average residual and the correlation between the residuals and $x$.

In [ ]:
other_line <- function(slope, intercept) {
  other <- heights |>
    mutate(Residual = Child - (slope * MidParent + intercept))
  cat("Average residual:", round(mean(other$Residual), 2), "\n")
  cat("Correlation with x:", round(cor(other$MidParent, other$Residual), 2), "\n")
  ggplot(other, aes(x = MidParent, y = Residual)) +
    geom_point(color = "red", alpha = 0.5) +
    geom_hline(yintercept = 0, color = "darkblue", linewidth = 2) +
    labs(title = "Residual Plot") +
    theme_minimal(base_size = 16)
}

The regression line:

In [ ]:
family_slope <- coef(family_model)[2]
family_intercept <- coef(family_model)[1]

other_line(family_slope, family_intercept)

The right slope, with the intercept 2 inches too low:

In [ ]:
other_line(family_slope, family_intercept - 2)

A slope that's too small (0.3), through the point of averages:

In [ ]:
other_line(0.3, mean(heights$Child) - 0.3 * mean(heights$MidParent))

## Part 4: The SD of the Residuals

The rmse of the regression line:

In [ ]:
sqrt(mean(heights$Residual^2))

The SD of the residuals:

In [ ]:
sd(heights$Residual)

$$\text{SD of residuals} = \sqrt{1 - r^2} \times \text{SD of } y$$

In [ ]:
family_r <- cor(heights$MidParent, heights$Child)
family_r

In [ ]:
sqrt(1 - family_r^2)

In [ ]:
sqrt(1 - family_r^2) * sd(heights$Child)

Compare with guessing the average height for every child:

In [ ]:
sd(heights$Child)

Districts: $r$ is higher, so the fraction is smaller.

In [ ]:
demographics_r <- cor(demographics$`College%`, demographics$`Median Income`)

c(demographics_r, sqrt(1 - demographics_r^2))

In [ ]:
c(sd(residuals(demographics_model)),
  sqrt(1 - demographics_r^2) * sd(demographics$`Median Income`))

`us_women`: $r$ is close to 1.

In [ ]:
women_r <- cor(us_women$height, us_women$weight)

c(women_r, sqrt(1 - women_r^2))

In [ ]:
c(sd(us_women$Residual), sd(us_women$weight))

## Part 5: Another Way to Interpret $r$

$$\frac{\text{SD of fitted values}}{\text{SD of } y} = |r|$$

In [ ]:
sd(heights$`Fitted Value`) / sd(heights$Child)

In [ ]:
family_r

Square it: $r^2$ is the fraction of the variance of $y$ that the line accounts for.

In [ ]:
c(var(heights$`Fitted Value`) / var(heights$Child), family_r^2)

## Practice: Penguins

Try this on your own after class.

Back to the penguins: predict body mass (`body_mass_g`, in grams) from flipper length (`flipper_length_mm`).

1. Add the residuals to `model_data`. Check that they average 0 and have correlation 0 with flipper length.
2. Find $r$. What fraction of the SD of body mass is the SD of the residuals?
3. Find the SD of the residuals two ways: with `sd()` and with the formula.
4. What fraction of the variance of body mass does the line account for?

**Done early?** Find the SD of the fitted values two ways.

In [ ]:
library(palmerpenguins)

# Keep rows with the two variables we need
model_data <- penguins |>
  filter(!is.na(flipper_length_mm), !is.na(body_mass_g)) |>
  select(species, flipper_length_mm, body_mass_g)

penguin_model <- lm(body_mass_g ~ flipper_length_mm, data = model_data)
coef(penguin_model)

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Done early? The SD of the fitted values